## Stage 4 – Data cleaning and quality checks

- `time` converted from text to datetime.
- No duplicate rows; one record per city and date.
- No missing days in 2021-01-01 to 2025-12-31 for any of the four cities.
- Logic checks passed: mean temperature always between min and max; rain_sum never above precipitation_sum; no negative precipitation; humidity within 0-100.
- Extreme values reviewed. Milan has 90.3 mm on 2025-04-16 and 88.2 mm on 2024-05-15, both inside multi-day heavy-rain episodes. They were kept: no technical error found. Limitation: reanalysis grid cells (~25 km) may shift timing or smooth local peaks compared with station data, so single-day extremes should be interpreted with care.
- Added columns: year, month, season (meteorological seasons).

In [22]:
PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

weather.to_csv(PROCESSED_DIR / "weather_clean.csv", index=False)
weather.shape

(7304, 12)

In [21]:
weather["year"] = weather["time"].dt.year
weather["month"] = weather["time"].dt.month


def get_season(month):
    """Return the meteorological season for a month number (1-12)."""
    if month in (12, 1, 2):
        return "Winter"
    if month in (3, 4, 5):
        return "Spring"
    if month in (6, 7, 8):
        return "Summer"
    return "Autumn"


weather["season"] = weather["month"].apply(get_season)
weather.groupby("season")["month"].unique()

season
Autumn    [9, 10, 11]
Spring      [3, 4, 5]
Summer      [6, 7, 8]
Winter     [1, 2, 12]
Name: month, dtype: object

In [20]:
import requests

units_response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 45.4642,
        "longitude": 9.19,
        "start_date": "2021-01-01",
        "end_date": "2021-01-02",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "rain_sum",
            "wind_speed_10m_max",
            "relative_humidity_2m_mean",
        ],
        "timezone": "auto",
    },
    timeout=30,
)
units_response.json()["daily_units"]

{'time': 'iso8601',
 'temperature_2m_mean': '°C',
 'precipitation_sum': 'mm',
 'rain_sum': 'mm',
 'wind_speed_10m_max': 'km/h',
 'relative_humidity_2m_mean': '%'}

In [19]:
weather["time"].dtype

dtype('<M8[us]')

In [18]:
milan_w[(milan_w["time"] >= "2024-05-14") & (milan_w["time"] <= "2024-05-18")][["time", "precipitation_sum", "rain_sum"]]

,time,precipitation_sum,rain_sum
3055,2024-05-14,10.8,10.8
3056,2024-05-15,88.2,88.2
3057,2024-05-16,40.9,40.9
3058,2024-05-17,0.9,0.9
3059,2024-05-18,6.0,6.0


In [17]:
milan_w[(milan_w["time"] >= "2025-04-14") & (milan_w["time"] <= "2025-04-18")][["time", "precipitation_sum", "rain_sum", "wind_speed_10m_max"]]

,time,precipitation_sum,rain_sum,wind_speed_10m_max
3390,2025-04-14,13.6,13.6,11.2
3391,2025-04-15,16.5,16.5,13.9
3392,2025-04-16,90.3,90.3,27.0
3393,2025-04-17,38.7,38.7,32.1
3394,2025-04-18,2.7,2.7,8.3


In [16]:
milan_w = weather[weather["city"] == "Milan"]
milan_w.nlargest(5, "precipitation_sum")[["time", "precipitation_sum", "rain_sum"]]

,time,precipitation_sum,rain_sum
3392,2025-04-16,90.3,90.3
3056,2024-05-15,88.2,88.2
2795,2023-08-28,73.4,73.4
2858,2023-10-30,63.2,63.2
3552,2025-09-23,60.0,60.0


In [15]:
weather.loc[weather.groupby("city")["wind_speed_10m_max"].idxmax(), ["city", "time", "wind_speed_10m_max"]]

,city,time,wind_speed_10m_max
1051,Astana,2023-11-18,52.9
5891,London,2022-02-18,68.0
2228,Milan,2022-02-07,54.5
3945,Paris,2021-10-21,52.4


In [14]:
weather.loc[weather.groupby("city")["precipitation_sum"].idxmax(), ["city", "time", "precipitation_sum"]]

,city,time,precipitation_sum
1598,Astana,2025-05-18,31.5
6839,London,2024-09-23,51.6
3392,Milan,2025-04-16,90.3
5029,Paris,2024-10-09,54.6


In [13]:
checks = {
    "mean temp below min temp": (weather["temperature_2m_mean"] < weather["temperature_2m_min"]).sum(),
    "mean temp above max temp": (weather["temperature_2m_mean"] > weather["temperature_2m_max"]).sum(),
    "rain_sum above precipitation_sum": (weather["rain_sum"] > weather["precipitation_sum"]).sum(),
    "negative precipitation": (weather["precipitation_sum"] < 0).sum(),
    "humidity outside 0-100": ((weather["relative_humidity_2m_mean"] < 0) | (weather["relative_humidity_2m_mean"] > 100)).sum(),
}
pd.Series(checks)

mean temp below min temp            0
mean temp above max temp            0
rain_sum above precipitation_sum    0
negative precipitation              0
humidity outside 0-100              0
dtype: int64

In [12]:
full_range = pd.date_range("2021-01-01", "2025-12-31", freq="D")

for city, group in weather.groupby("city"):
    missing_days = full_range.difference(group["time"])
    print(city, "missing days:", len(missing_days))

Astana missing days: 0
London missing days: 0
Milan missing days: 0
Paris missing days: 0


In [11]:
weather.duplicated(subset=["city", "time"]).sum()

np.int64(0)

In [10]:
weather.duplicated().sum()

np.int64(0)

In [9]:
weather["time"] = pd.to_datetime(weather["time"])
weather["time"].dtype

dtype('<M8[us]')

## Stage 3 – Data exploration

- Source: Open-Meteo Historical Weather API (reanalysis data, not station readings).
- Four cities (Astana, Milan, Paris, London), 1826 daily rows each (2021-01-01 to 2025-12-31).
- 9 columns: city, time, 3 temperature variables, precipitation_sum, rain_sum, wind_speed_10m_max, relative_humidity_2m_mean.
- No missing values in any column.
- `time` is stored as text (str) and must be converted to a date type in Stage 4.
- Value ranges look plausible at first sight; detailed quality checks follow in Stage 4.

In [8]:
weather.groupby("city")[["temperature_2m_mean", "precipitation_sum", "wind_speed_10m_max", "relative_humidity_2m_mean"]].agg(["min", "max"])

temperature_2m_mean       precipitation_sum       wind_speed_10m_max  \
                       min   max               min   max                min   
city                                                                          
Astana               -31.3  30.5               0.0  31.5                6.6   
London                -4.8  29.1               0.0  51.6                3.5   
Milan                 -1.0  31.7               0.0  90.3                3.3   
Paris                 -3.4  32.0               0.0  54.6                4.1   

             relative_humidity_2m_mean      
         max                       min max  
city                                        
Astana  52.9                        25  98  
London  68.0                        37  97  
Milan   54.5                        28  99  
Paris   52.4                        34  98

In [7]:
weather.isna().sum()

city                         0
time                         0
temperature_2m_mean          0
temperature_2m_max           0
temperature_2m_min           0
precipitation_sum            0
rain_sum                     0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
dtype: int64

In [6]:
weather["city"].value_counts()

city
Astana    1826
Milan     1826
Paris     1826
London    1826
Name: count, dtype: int64

In [5]:
cities = ["astana", "milan", "paris", "london"]
frames = [pd.read_csv(RAW_DIR / f"{c}_daily_2021_2025.csv") for c in cities]
weather = pd.concat(frames, ignore_index=True)
weather.shape

(7304, 9)

In [2]:
milan.describe()

,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,wind_speed_10m_max,relative_humidity_2m_mean
count,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000
mean,14.370756,19.046112,10.188335,3.547536,3.499836,13.030449,74.338445
std,8.072440,8.555091,7.603399,8.319326,8.244625,5.705549,14.508360
min,-1.000000,1.700000,-5.300000,0.000000,0.000000,3.300000,28.000000
25%,7.600000,11.800000,3.700000,0.000000,0.000000,9.000000,65.000000
50%,14.200000,18.600000,10.200000,0.000000,0.000000,11.800000,76.000000
75%,21.375000,26.200000,16.875000,2.500000,2.475000,15.775000,86.000000
max,31.700000,37.700000,26.400000,90.300000,90.300000,54.500000,99.000000


In [3]:
milan.info()

<class 'pandas.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   city                       1826 non-null   str    
 1   time                       1826 non-null   str    
 2   temperature_2m_mean        1826 non-null   float64
 3   temperature_2m_max         1826 non-null   float64
 4   temperature_2m_min         1826 non-null   float64
 5   precipitation_sum          1826 non-null   float64
 6   rain_sum                   1826 non-null   float64
 7   wind_speed_10m_max         1826 non-null   float64
 8   relative_humidity_2m_mean  1826 non-null   int64  
dtypes: float64(6), int64(1), str(2)
memory usage: 128.5 KB


In [4]:
milan.shape

(1826, 9)

In [1]:
from pathlib import Path

import pandas as pd

# The notebook lives in notebooks/, so the data is one folder up
RAW_DIR = Path("..") / "data" / "raw"

milan = pd.read_csv(RAW_DIR / "milan_daily_2021_2025.csv")
milan.head()

,city,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,wind_speed_10m_max,relative_humidity_2m_mean
0,Milan,2021-01-01,0.9,1.7,-0.9,19.7,13.3,12.2,96
1,Milan,2021-01-02,2.4,4.0,1.6,12.2,12.2,13.7,93
2,Milan,2021-01-03,4.3,6.5,3.0,9.0,9.0,18.0,95
3,Milan,2021-01-04,4.2,5.8,2.5,5.9,5.9,14.7,93
4,Milan,2021-01-05,4.0,6.2,2.0,7.3,7.3,12.1,94
